In [0]:
# Purpose: Gold layer - governance scoring, structure, profiling, executive KPIs,
#          granular BI summaries, structural-consistency (Objective 1), and the
#          defensible region-based access-control subsystem (Genie-facing).
# Author: Erzen Citaku - Metadata & Catalog Engineer (R4)
# Last updated: 2026-06-16
# Dependencies: dbx_metadata_governance_dev.silver.metadata_validated (rubric 2.0)
# Notes:
#   - Reads the rebuilt, deduplicated Silver. Keys are SYSTEM-SCOPED: the same
#     database/schema/table/column name in different source systems is a DISTINCT
#     asset, so counts are per-system instances, not distinct names. Expect
#     ~60 databases, ~180 schemas, ~1.7k tables, ~7.1k columns.
#   - Objective 1 structural consistency uses a DYNAMIC modal standard column
#     count (overridable via spark.conf "structural.standard_column_count") with
#     a +/- 30% tolerance band.
#   - Access control derives region from an explicit per-database region_assignment
#     policy (NOT the random source location tag). current_user() is excluded from
#     all tables and must be applied in a query-time view.

import dlt
from pyspark.sql import functions as F
from pyspark.sql.window import Window

STRUCTURAL_TOLERANCE = 0.30  # +/- 30% band around the modal standard column count


def silver():
    return spark.read.table("dbx_metadata_governance_dev.silver.metadata_validated")


def _check(name):
    # Silver check columns are nullable booleans; treat NULL as not-passed.
    return F.coalesce(F.col(f"check_{name}"), F.lit(False))


def metadata_base():
    return silver()


# ============================================================================
# Governance scoring
# ============================================================================

@dlt.table(name="scores_pre", comment="Gold: pre-calculated governance scoring columns")
def scores_pre():
    df = metadata_base()
    return (
        df
        .withColumn("documentation_score", F.round(F.col("completeness_pct"), 2))
        .withColumn("quality_score", F.round(100 - F.coalesce(F.col("dq_invalid_pct"), F.lit(0.0)), 2))
        .withColumn("classification_score", F.when(_check("security_classification_present"), 100).otherwise(0))
        .withColumn(
            "overall_governance_score",
            F.round(
                (F.col("completeness_pct") * 0.50) +
                ((100 - F.coalesce(F.col("dq_invalid_pct"), F.lit(0.0))) * 0.30) +
                (F.when(_check("security_classification_present"), 100).otherwise(0) * 0.20),
                2
            )
        )
    )


@dlt.table(name="table_governance", comment="Gold: governance score and column rollup by table")
def table_governance():
    df = dlt.read("scores_pre")
    return (
        df.groupBy(
            "system_name", "database_name", "schema_name", "table_id", "table_name", "table_desc", "data_steward"
        )
        .agg(
            F.countDistinct("column_id").alias("column_count"),
            F.round(F.avg("overall_governance_score"), 2).alias("table_governance_score"),
            F.round(F.avg("documentation_score"), 2).alias("avg_documentation_score"),
            F.round(F.avg("quality_score"), 2).alias("avg_quality_score"),
            F.sum(F.col("pii_flag").cast("int")).alias("pii_column_count"),
            F.sum(F.col("critical_data_element_flag").cast("int")).alias("critical_column_count")
        )
    )


# ============================================================================
# Structure & profile
# ============================================================================

@dlt.table(name="table_structure", comment="Gold: database, schema, table, and column structure")
def table_structure():
    df = metadata_base()
    return df.select(
        "database_id", "database_name", "system_name", "location",
        "schema_id", "schema_name",
        "table_id", "table_name", "table_desc",
        "column_id", "column_name", "column_desc"
    )


@dlt.table(name="profile", comment="Gold: metadata profile for columns")
def profile():
    df = metadata_base()
    return df.select(
        "system_name",
        "database_name", "schema_name", "table_name",
        "column_id", "column_name",
        "pii_flag", "critical_data_element_flag", "security_classification",
        "certification_level", "achieved_cert_level",
        "term_name", "term_description", "term_subdomain",
        "data_steward", "tag_value",
        "total_record_count", "invalid_record_count",
        "dq_invalid_pct", "completeness_pct"
    )


@dlt.table(name="dlt_summary", comment="Gold: pipeline data-quality summary")
def dlt_summary():
    df = metadata_base()
    return df.agg(
        F.count("*").alias("silver_valid_rows"),
        F.sum(F.when(_check("invalid_record_ratio_within_threshold"), 1).otherwise(0)).alias("dq_pass_rows"),
        F.sum(F.when(~_check("invalid_record_ratio_within_threshold"), 1).otherwise(0)).alias("dq_failed_rows"),
        F.sum(F.when(_check("sensitivity_flags_set"), 1).otherwise(0)).alias("pii_consistent_rows"),
        F.sum(F.when(~_check("sensitivity_flags_set"), 1).otherwise(0)).alias("pii_inconsistent_rows")
    )


# ============================================================================
# Executive + granular KPI summaries
# ============================================================================

@dlt.table(name="kpi_summary", comment="Gold: executive governance KPI summary with certification distribution")
def kpi_summary():
    df = metadata_base()
    agg = df.agg(
        F.count("*").alias("total_columns"),
        F.countDistinct("table_id").alias("total_tables"),
        F.countDistinct("schema_id").alias("total_schemas"),
        F.countDistinct("database_id").alias("total_databases"),
        F.round(F.avg("completeness_pct"), 2).alias("avg_completeness_pct"),
        F.round(F.avg("dq_invalid_pct"), 2).alias("avg_dq_invalid_pct"),
        F.sum(F.col("pii_flag").cast("int")).alias("pii_columns"),
        F.sum(F.col("critical_data_element_flag").cast("int")).alias("critical_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "Certified", 1).otherwise(0)).alias("certified_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "Documented", 1).otherwise(0)).alias("documented_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "Registered", 1).otherwise(0)).alias("registered_columns"),
        F.sum(F.when(F.col("achieved_cert_level") == "None", 1).otherwise(0)).alias("unclassified_columns"),
    )
    return (
        agg
        .withColumn("avg_columns_per_table",
                    F.round(F.col("total_columns") / F.col("total_tables"), 1))
        .withColumn("certified_pct", F.round(F.col("certified_columns") / F.col("total_columns") * 100, 1))
        .withColumn("documented_pct", F.round(F.col("documented_columns") / F.col("total_columns") * 100, 1))
        .withColumn("registered_pct", F.round(F.col("registered_columns") / F.col("total_columns") * 100, 1))
        .withColumn("unclassified_pct", F.round(F.col("unclassified_columns") / F.col("total_columns") * 100, 1))
    )


@dlt.table(name="cert_level_summary", comment="Gold: certification distribution per table")
def cert_level_summary():
    df = metadata_base()
    return (
        df.groupBy("system_name", "database_name", "schema_name", "table_id", "table_name")
        .agg(
            F.count("*").alias("column_count"),
            F.sum(F.when(F.col("achieved_cert_level") == "Certified", 1).otherwise(0)).alias("certified_columns"),
            F.sum(F.when(F.col("achieved_cert_level") == "Documented", 1).otherwise(0)).alias("documented_columns"),
            F.sum(F.when(F.col("achieved_cert_level") == "Registered", 1).otherwise(0)).alias("registered_columns"),
            F.sum(F.when(F.col("achieved_cert_level") == "None", 1).otherwise(0)).alias("unclassified_columns"),
        )
        .withColumn("certified_pct", F.round(F.col("certified_columns") / F.col("column_count") * 100, 1))
        .withColumn("documented_pct", F.round(F.col("documented_columns") / F.col("column_count") * 100, 1))
        .withColumn("registered_pct", F.round(F.col("registered_columns") / F.col("column_count") * 100, 1))
        .withColumn("unclassified_pct", F.round(F.col("unclassified_columns") / F.col("column_count") * 100, 1))
    )


@dlt.table(name="pii_classification_summary", comment="Gold: security-classification and sensitivity coverage rates")
def pii_classification_summary():
    df = metadata_base().withColumn(
        "classification", F.coalesce(F.col("security_classification"), F.lit("Unclassified"))
    )
    total = df.count()
    return (
        df.groupBy("classification")
        .agg(
            F.count("*").alias("column_count"),
            F.sum(F.col("pii_flag").cast("int")).alias("pii_columns"),
            F.sum(F.col("critical_data_element_flag").cast("int")).alias("critical_columns"),
            F.sum(F.when(_check("sensitivity_flags_set"), 1).otherwise(0)).alias("sensitivity_set_columns"),
        )
        .withColumn("pct_of_catalog", F.round(F.col("column_count") / F.lit(total) * 100, 1))
        .withColumn("sensitivity_coverage_pct",
                    F.round(F.col("sensitivity_set_columns") / F.col("column_count") * 100, 1))
    )


@dlt.table(name="domain_summary", comment="Gold: governance metrics grouped by business domain")
def domain_summary():
    df = dlt.read("scores_pre").withColumn(
        "domain", F.coalesce(F.col("tag_value"), F.col("term_subdomain"), F.lit("Unassigned"))
    )
    return (
        df.groupBy("domain")
        .agg(
            F.count("*").alias("column_count"),
            F.countDistinct("table_id").alias("table_count"),
            F.round(F.avg("overall_governance_score"), 2).alias("avg_governance_score"),
            F.sum(F.when(F.col("achieved_cert_level") == "Certified", 1).otherwise(0)).alias("certified_columns"),
            F.sum(F.when(F.col("achieved_cert_level") == "Documented", 1).otherwise(0)).alias("documented_columns"),
            F.sum(F.when(F.col("achieved_cert_level") == "Registered", 1).otherwise(0)).alias("registered_columns"),
            F.sum(F.when(F.col("achieved_cert_level") == "None", 1).otherwise(0)).alias("unclassified_columns"),
        )
        .withColumn("certified_pct", F.round(F.col("certified_columns") / F.col("column_count") * 100, 1))
    )


# ============================================================================
# Objective 1: structural consistency
# ============================================================================

@dlt.table(name="structural_consistency", comment="Gold: per-table column-count consistency vs the catalog standard (Objective 1)")
def structural_consistency():
    df = metadata_base()
    per_table = (
        df.groupBy("system_name", "database_id", "schema_id", "table_id", "database_name", "schema_name", "table_name")
          .agg(F.countDistinct("column_id").alias("column_count"))
    )

    # Standard column count: spark.conf override, else dynamic modal value.
    try:
        std_override = int(spark.conf.get("structural.standard_column_count"))
    except Exception:
        std_override = None

    if std_override is not None:
        per_table = per_table.withColumn("standard_column_count", F.lit(std_override))
    else:
        modal = (
            per_table.groupBy("column_count").count()
            .orderBy(F.col("count").desc(), F.col("column_count").asc())
            .limit(1)
            .select(F.col("column_count").alias("standard_column_count"))
        )
        per_table = per_table.crossJoin(modal)

    return (
        per_table
        .withColumn("lower_bound", F.round(F.col("standard_column_count") * (1 - STRUCTURAL_TOLERANCE)))
        .withColumn("upper_bound", F.round(F.col("standard_column_count") * (1 + STRUCTURAL_TOLERANCE)))
        .withColumn(
            "structural_status",
            F.when(F.col("column_count") < F.col("lower_bound"), F.lit("Outlier: too few columns"))
             .when(F.col("column_count") > F.col("upper_bound"), F.lit("Outlier: too many columns"))
             .otherwise(F.lit("Compliant"))
        )
        .withColumn("is_outlier", F.col("structural_status") != "Compliant")
        .withColumn(
            "corrective_action",
            F.when(
                F.col("column_count") < F.col("lower_bound"),
                F.concat(
                    F.lit("Table '"), F.col("table_name"), F.lit("' has "), F.col("column_count"),
                    F.lit(" columns vs the standard of "), F.col("standard_column_count"),
                    F.lit(". Verify no columns were dropped during ingestion and that all source fields are catalogued.")
                )
            ).when(
                F.col("column_count") > F.col("upper_bound"),
                F.concat(
                    F.lit("Table '"), F.col("table_name"), F.lit("' has "), F.col("column_count"),
                    F.lit(" columns vs the standard of "), F.col("standard_column_count"),
                    F.lit(". Review for redundant, duplicated, or mis-assigned columns and consider normalizing.")
                )
            ).otherwise(F.lit("No action required; table is within the structural standard."))
        )
    )


@dlt.table(name="structural_summary", comment="Gold: catalog-level structural-consistency KPIs (Objective 2 dashboard)")
def structural_summary():
    sc = dlt.read("structural_consistency")
    return (
        sc.agg(
            F.count("*").alias("total_tables"),
            F.max("standard_column_count").alias("standard_column_count"),
            F.round(F.avg("column_count"), 1).alias("avg_columns_per_table"),
            F.sum(F.when(~F.col("is_outlier"), 1).otherwise(0)).alias("compliant_tables"),
            F.sum(F.when(F.col("is_outlier"), 1).otherwise(0)).alias("outlier_tables"),
        )
        .withColumn("structural_consistency_pct",
                    F.round(F.col("compliant_tables") / F.col("total_tables") * 100, 1))
    )


@dlt.table(name="governance_gaps", comment="Gold: missing governance metadata and failed checks")
def governance_gaps():
    df = metadata_base()
    return (
        df
        .withColumn(
            "gap_reason",
            F.concat_ws(
                ", ",
                F.when(~_check("column_description_present"), "Missing column description"),
                F.when(~_check("business_term_linked"), "Missing business term"),
                F.when(~_check("security_classification_present"), "Missing or invalid security classification"),
                F.when(~_check("data_steward_assigned"), "Missing data steward"),
                F.when(~_check("domain_tag_present"), "Missing domain tag"),
                F.when(~_check("sensitivity_flags_set"), "PII or CDE flag is null"),
                F.when(~_check("certification_level_populated"), "Missing or invalid certification level"),
                F.when(~_check("schema_assignment_valid"), "Invalid schema assignment"),
                F.when(~_check("invalid_record_ratio_within_threshold"), "Data quality failed: invalid record ratio exceeds 5%"),
            )
        )
        .filter(F.col("gap_reason") != "")
        .select(
            "database_name", "schema_name", "table_name",
            "column_id", "column_name", "data_steward",
            "security_classification", "achieved_cert_level",
            "completeness_pct", "dq_invalid_pct", "gap_reason"
        )
    )


# ============================================================================
# Access-control subsystem (Genie-facing)
# Region is an explicit POLICY (per database), not inferred from the random
# source location tag. current_user() is intentionally excluded from every
# table; apply it at query time in a Gold view.
# ============================================================================

@dlt.table(name="region_assignment", comment="Gold: explicit, version-controlled data-residency policy per database")
def region_assignment():
    return spark.createDataFrame(
        [
            # database, asset_region, required_access_group, allowed_region, policy_rationale
            ("finance_dw",   "EU", "EU_EMPLOYEES", "Europe", "Financial data subject to EU data-residency rules."),
            ("finance_mart", "EU", "EU_EMPLOYEES", "Europe", "Financial mart derived from EU-resident sources."),
            ("billing",      "EU", "EU_EMPLOYEES", "Europe", "Billing records contain EU customer financial data."),
            ("erp",          "EU", "EU_EMPLOYEES", "Europe", "ERP data governed under EU operational data rules."),
            ("crm",          "US", "US_EMPLOYEES", "US", "US customer PII held under US handling requirements."),
            ("customer360",  "US", "US_EMPLOYEES", "US", "Unified US customer profiles containing PII."),
            ("cdp",          "US", "US_EMPLOYEES", "US", "Customer data platform with US PII."),
            ("hr_mart",      "US", "US_EMPLOYEES", "US", "US HR and payroll data."),
            ("people_ops",   "US", "US_EMPLOYEES", "US", "US people-operations and employee records."),
            ("analytics",    "UNRESTRICTED", "ALL_EMPLOYEES", "Any", "Aggregated, non-PII analytical outputs."),
            ("lakehouse",    "UNRESTRICTED", "ALL_EMPLOYEES", "Any", "Shared platform and reference data."),
            ("core_db",      "UNRESTRICTED", "ALL_EMPLOYEES", "Any", "Non-sensitive core reference data."),
        ],
        ["database_name", "asset_region", "required_access_group", "allowed_region", "policy_rationale"]
    )


@dlt.table(name="user_profile", comment="Gold: user profile data for access eligibility checks")
def user_profile():
    return spark.createDataFrame(
        [
            ("gegedobruna@gmail.com", "Hungary", "Europe", "Budapest", "Data Engineering", "EU_EMPLOYEES"),
            ("ajete.isaku123@gmail.com", "Germany", "Europe", "Berlin", "Data Governance", "EU_EMPLOYEES"),
            ("erza.ademii24@gmail.com", "Austria", "Europe", "Vienna", "Metadata Management", "EU_EMPLOYEES"),
            ("bledirexha05@outlook.com", "Switzerland", "Europe", "Zurich", "Platform Engineering", "EU_EMPLOYEES"),
            ("erzencitaku@gmail.com", "United States", "US", "New York", "Data Governance", "US_EMPLOYEES"),
            ("flandercanaj18@gmail.com", "United States", "US", "Chicago", "Data Quality", "US_EMPLOYEES"),
            ("gresahasani19@gmail.com", "United States", "US", "Dallas", "Metadata Management", "US_EMPLOYEES"),
            ("ademierza14@gmail.com", "United States", "US", "San Francisco", "Platform Engineering", "US_EMPLOYEES"),
        ],
        ["user_email", "user_country", "user_region", "employee_location", "department", "access_group"]
    )


@dlt.table(name="asset_access_policy", comment="Gold: per-table access policy, inherited from each table's database region_assignment")
def asset_access_policy():
    tables = (
        metadata_base()
        .select("table_id", "database_name", "schema_name", "table_name")
        .dropDuplicates(["table_id"])
    )
    region = dlt.read("region_assignment")
    return (
        tables.join(region, "database_name", "left")
        # Databases without an explicit rule fall back to UNRESTRICTED and are flagged.
        .withColumn("is_default_policy", F.col("asset_region").isNull())
        .withColumn("asset_region", F.coalesce(F.col("asset_region"), F.lit("UNRESTRICTED")))
        .withColumn("required_access_group", F.coalesce(F.col("required_access_group"), F.lit("ALL_EMPLOYEES")))
        .withColumn("allowed_region", F.coalesce(F.col("allowed_region"), F.lit("Any")))
        .withColumn("policy_rationale", F.coalesce(
            F.col("policy_rationale"),
            F.lit("No explicit region policy for this database; defaulted to UNRESTRICTED. Governance should add a rule.")
        ))
        .select(
            "table_id", "database_name", "schema_name", "table_name",
            "asset_region", "required_access_group", "allowed_region",
            "policy_rationale", "is_default_policy"
        )
    )


@dlt.table(name="asset_access_check", comment="Gold: resolved access decision by user and asset for Genie")
def asset_access_check():
    users_df = dlt.read("user_profile").alias("u")
    policies_df = dlt.read("asset_access_policy").alias("a")

    return (
        policies_df
        .crossJoin(users_df)
        .withColumn(
            "asset_location",
            F.concat_ws(".", F.col("a.database_name"), F.col("a.schema_name"), F.col("a.table_name"))
        )
        .withColumn(
            "access_decision",
            # UNRESTRICTED -> everyone; otherwise region AND access group must match.
            F.when(F.col("a.asset_region") == "UNRESTRICTED", F.lit("Eligible"))
             .when(
                 (F.col("u.user_region") == F.col("a.allowed_region")) &
                 (F.col("u.access_group") == F.col("a.required_access_group")),
                 F.lit("Eligible")
             )
             .otherwise(F.lit("Not Eligible"))
        )
        .withColumn(
            "access_reason",
            F.when(F.col("a.asset_region") == "UNRESTRICTED",
                   F.lit("Asset is unrestricted; available to all employees."))
             .when(F.col("access_decision") == "Eligible",
                   F.lit("User's region and access group match the asset's data-residency policy."))
             .otherwise(
                   F.lit("Asset is restricted to its region; user's region or access group does not match."))
        )
        .select(
            F.col("asset_location"),
            F.col("a.table_id").alias("table_id"),
            F.col("a.table_name").alias("table_name"),
            F.col("a.asset_region").alias("asset_region"),
            F.col("u.user_email").alias("user_email"),
            F.col("u.user_region").alias("user_region"),
            F.col("u.department").alias("department"),
            F.col("a.required_access_group").alias("required_access_group"),
            F.col("u.access_group").alias("user_access_group"),
            F.col("a.allowed_region").alias("allowed_region"),
            F.col("a.policy_rationale").alias("policy_rationale"),
            F.col("a.is_default_policy").alias("is_default_policy"),
            F.col("access_decision"),
            F.col("access_reason")
        )
    )